# Embedding Model Comparison for RAG: MiniLM vs E5 vs BGE

**Document:** `sample-sdf-document.pdf` (Cytiva storage-conditions letter + two Certificates of Quality, 3 pages)
**Stack:** PyMuPDF (parsing) + LlamaIndex (chunking/indexing/retrieval) + HuggingFace embeddings + Gemini (generation)
**Author:** Bashar

This notebook takes the RAG pipeline from the previous project and changes **one variable only: the embedding model**. The parsing, chunking, retriever settings, LLM and queries are held constant, so any difference in the retrieved chunks or the generated answers is attributable to the embedding model.

Each model is scored on the four criteria from the task:

| Criterion | How it is measured here |
|---|---|
| Context relevance | Does the top-k contain the chunk holding the known answer? At what rank? |
| Answer quality | Gemini's answer generated from each model's retrieved chunks |
| Speed | Time to embed all chunks (index build) and mean time per query |
| Consistency | Whether the model finds the right chunk across all three queries |

## Step 0: Install dependencies

In [ ]:
%pip install -q pymupdf llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface google-genai nest_asyncio

In [1]:
import nest_asyncio
nest_asyncio.apply()  # Colab already runs an event loop; LlamaIndex needs to nest inside it

## Step 1: Gemini API key (used for answer generation)

Add the key as a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar), or paste it below. If no key is found the notebook still runs: it compares retrieval only and skips the generated answers.

In [2]:
import os

GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # or leave as-is and use a Colab secret named GOOGLE_API_KEY

if GOOGLE_API_KEY == "YOUR_API_KEY_HERE":
    try:
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

USE_LLM = bool(GOOGLE_API_KEY and GOOGLE_API_KEY.strip())
if USE_LLM:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("Gemini key found: answers will be generated.")
else:
    print("No Gemini key found: running retrieval-only (no generated answers).")
    print("Get a free key at https://aistudio.google.com/app/apikey")

Gemini key found: answers will be generated.


## Step 2: Upload the PDF

In [3]:
import glob

candidates = glob.glob("*sdf-document*.pdf") or glob.glob("/content/*sdf-document*.pdf")
if not candidates:
    from google.colab import files
    print("Please select 'sample-sdf-document.pdf' to upload:")
    uploaded = files.upload()
    candidates = list(uploaded.keys())

pdf_path = candidates[0]
print("Using PDF:", pdf_path)

Using PDF: sample-sdf-document.pdf


## Step 3: Parse and chunk (held constant for every model)

Parsing is the same as the previous project: PyMuPDF, one `Document` per page, tagged with its page number.

**Chunking is deliberately different from the previous project.** There I used page-level chunks, which gave 3 chunks for 3 pages. With only 3 chunks and `top_k=3`, every embedding model would retrieve the whole document and the comparison would show nothing. Here the pages are split into smaller chunks (128 tokens, 20 overlap) so the retriever has to actually *choose*, and a weaker embedding model has room to choose wrong.

In [4]:
import fitz  # PyMuPDF
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 128
CHUNK_OVERLAP = 20
TOP_K = 2

def load_pdf_with_pymupdf(path):
    # Parse a PDF into one LlamaIndex Document per page.
    doc = fitz.open(path)
    documents = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if not text.strip():
            continue
        documents.append(Document(
            text=text,
            metadata={"page_number": i + 1},
            # keep metadata out of the embedded text so only the document content is compared
            excluded_embed_metadata_keys=["page_number"],
            excluded_llm_metadata_keys=[],
        ))
    doc.close()
    return documents

pdf_docs = load_pdf_with_pymupdf(pdf_path)
splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(pdf_docs)

print(f"{len(pdf_docs)} pages -> {len(nodes)} chunks (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")
for i, n in enumerate(nodes):
    print(f"chunk {i:>2} | page {n.metadata['page_number']} | {len(n.get_content().split()):>3} words | {n.get_content()[:70].replace(chr(10), ' ')}...")

3 pages -> 12 chunks (chunk_size=128, overlap=20)

chunk  0 | page 1 |  76 words | Cytiva 100 Results Way Marlborough, MA 01752 United States Page 1 / 1 ...
chunk  1 | page 1 |  59 words | Extended storage below the recommended +5 could lead to  brittleness o...
chunk  2 | page 1 |  61 words | Description Part Number Operating Temperature High Flow Kit F, Modifie...
chunk  3 | page 1 |   7 words | Best regards, Mike Toner Product Manager michaeltoner@cytiva.com...
chunk  4 | page 2 |  71 words | Certificate of Quality  This product is manufactured in compliance wit...
chunk  5 | page 2 |  32 words | cytiva.com  29190939 Rev AD Product: ÄKTA™ ready Gradient Flow Section...
chunk  6 | page 2 |  44 words | Modified, ÄKTATM ready Expiration Date:  20230126  Product Release Cri...
chunk  7 | page 2 |  56 words | Description  Limits  Autoclave Pump tubing 121°C > 15 min  Gamma Irrad...
chunk  8 | page 3 |  37 words | Cytiva cytiva.com Certificate of Quality This product is manufactured ...
c

## Step 4: Test queries and ground truth

The same three queries are used for every model. Each one has a known answer in the document, so "did it retrieve the right chunk?" can be checked automatically instead of by eye: a retrieved chunk counts as **relevant** if it contains the answer string.

| # | Query | Answer lives in | Answer string checked |
|---|---|---|---|
| Q1 | Storage conditions | Page 1 letter | `storage temperature` |
| Q2 | QC test methods | Page 3 certificate (test table) | `SOP-QC-042` |
| Q3 | Bio-safety / regulatory standards | Page 2 and 3 certificates | `Pharmacopeia` |

Q1 and Q2 are the prompts from the previous project. Q3 is worded differently from the document ("standards ... comply with" vs. the document's "Regulatory Conformance / Bio-safety") to test semantic matching rather than keyword overlap.

In [5]:
QUERIES = [
    {"id": "Q1", "query": "What are the storage conditions for this product?",
     "answer_strings": ["storage temperature"]},
    {"id": "Q2", "query": "What test methods were used for quality control?",
     "answer_strings": ["SOP-QC-042"]},
    {"id": "Q3", "query": "Which bio-safety and regulatory standards do the materials comply with?",
     "answer_strings": ["Pharmacopeia"]},
]

def is_relevant(text, answer_strings):
    return any(s.lower() in text.lower() for s in answer_strings)

# Sanity check: every query must have at least one chunk that contains its answer
for q in QUERIES:
    n_rel = sum(is_relevant(n.get_content(), q["answer_strings"]) for n in nodes)
    print(f"{q['id']}: {n_rel} of {len(nodes)} chunks contain the answer")
    assert n_rel > 0, f"No chunk contains the answer for {q['id']}"

Q1: 1 of 12 chunks contain the answer
Q2: 1 of 12 chunks contain the answer
Q3: 2 of 12 chunks contain the answer


## Step 5: Run the comparison

For each embedding model the index is rebuilt from scratch, so documents and queries are embedded in the same vector space. Three things are timed separately:

- **Load time**: loading the model (first run includes the download, so it is reported but *not* used to judge speed)
- **Index time**: embedding all chunks
- **Query time**: mean of 5 retrievals per query

**Instruction prefixes.** E5 and BGE were trained with instruction prefixes: E5 expects `query: ` in front of queries and `passage: ` in front of documents, and BGE expects a "Represent this sentence for searching relevant passages: " prefix on queries. MiniLM uses none. Just swapping `model_name` (as the module suggests) does not add them: when I printed `embed_model.query_instruction` after a plain swap it was `None` for all three models. They are therefore passed explicitly below, so each model is tested the way its model card says it should be used.

In [6]:
import time
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# (model name, query prefix, passage prefix) -- prefixes taken from each model's HuggingFace model card
EMBEDDING_MODELS = {
    "MiniLM": ("sentence-transformers/all-MiniLM-L6-v2", None, None),
    "E5": ("intfloat/e5-small-v2", "query: ", "passage: "),
    "BGE": ("BAAI/bge-small-en-v1.5", "Represent this sentence for searching relevant passages: ", None),
}

if USE_LLM:
    from llama_index.llms.google_genai import GoogleGenAI
    from llama_index.core import get_response_synthesizer
    llm = GoogleGenAI(model="models/gemini-3.8-flash", temperature=0)
    Settings.llm = llm
    synthesizer = get_response_synthesizer(llm=llm)
else:
    Settings.llm = None

def generate_answer(query, retrieved, retries=5):
    # Gemini's free tier occasionally returns 503 "high demand"; wait and retry instead of failing the run
    for attempt in range(retries):
        try:
            return str(synthesizer.synthesize(query, nodes=retrieved)).strip()
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 10 * (attempt + 1)
            print(f"  Gemini call failed ({type(e).__name__}); retrying in {wait}s")
            time.sleep(wait)

N_TIMING_RUNS = 5
results = {}

for label, (model_name, query_prefix, passage_prefix) in EMBEDDING_MODELS.items():
    print(f"\n{'=' * 90}\n{label}: {model_name}\n{'=' * 90}")

    t0 = time.perf_counter()
    embed_model = HuggingFaceEmbedding(
        model_name=model_name, query_instruction=query_prefix, text_instruction=passage_prefix)
    load_s = time.perf_counter() - t0
    print(f"query prefix: {query_prefix!r} | passage prefix: {passage_prefix!r}")

    t0 = time.perf_counter()
    index = VectorStoreIndex(nodes, embed_model=embed_model)
    index_s = time.perf_counter() - t0

    retriever = index.as_retriever(similarity_top_k=TOP_K)
    # full ranking over every chunk, used only to find the rank of the first relevant chunk
    full_retriever = index.as_retriever(similarity_top_k=len(nodes))

    per_query = []
    for q in QUERIES:
        t0 = time.perf_counter()
        for _ in range(N_TIMING_RUNS):
            retrieved = retriever.retrieve(q["query"])
        query_ms = (time.perf_counter() - t0) / N_TIMING_RUNS * 1000

        full_ranking = full_retriever.retrieve(q["query"])
        first_rel_rank = next(
            (r for r, n in enumerate(full_ranking, 1) if is_relevant(n.get_content(), q["answer_strings"])), None)

        answer = None
        if USE_LLM:
            answer = generate_answer(q["query"], retrieved)

        per_query.append({
            "id": q["id"], "query": q["query"], "query_ms": query_ms,
            "first_relevant_rank": first_rel_rank,
            "hit": first_rel_rank is not None and first_rel_rank <= TOP_K,
            "chunks": [{"rank": r, "score": n.score, "page": n.metadata.get("page_number"),
                        "relevant": is_relevant(n.get_content(), q["answer_strings"]),
                        "text": n.get_content()} for r, n in enumerate(retrieved, 1)],
            "answer": answer,
        })

        print(f"\n--- {q['id']}: {q['query']}")
        for c in per_query[-1]["chunks"]:
            flag = "RELEVANT" if c["relevant"] else "not relevant"
            print(f"  [{c['rank']}] score={c['score']:.4f} page={c['page']} ({flag})")
            print("      " + c["text"][:220].replace("\n", " | "))
        print(f"  first relevant chunk at rank {first_rel_rank} of {len(nodes)}")
        if answer:
            print(f"  ANSWER: {answer}")

    results[label] = {"model_name": model_name, "load_s": load_s, "index_s": index_s, "queries": per_query}

print("\nDone.")


MiniLM: sentence-transformers/all-MiniLM-L6-v2


query prefix: None | passage prefix: None



--- Q1: What are the storage conditions for this product?
  [1] score=0.2776 page=1 (RELEVANT)
      Cytiva | 100 Results Way | Marlborough, MA 01752 | United States | Page 1 / 1 | cytiva.com | 3 June, 2022 | Re: ÄKTATM ready Flow Kit Storage Conditions | To Whom It May Concern, | The recommended storage temperature for standard ÄKTA rea
  [2] score=0.1866 page=2 (not relevant)
      Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulatory Conformance  | Bio-safety | All polymeric materials in contact with the process fluid comply with United State
  first relevant chunk at rank 1 of 12
  ANSWER: The recommended storage temperature is greater than +5°C (> +5°C).



--- Q2: What test methods were used for quality control?
  [1] score=0.3076 page=3 (RELEVANT)
      0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visual Inspection | No defects visible | Pass | Package Integrity | Sealed, no damage | Pass | Regulatory Conformance | Bio-safet
  [2] score=0.2737 page=3 (not relevant)
      Cytiva | cytiva.com | Certificate of Quality | This product is manufactured in compliance with our ISO 9001 certified quality management system. | Issued by Cytiva Westborough Quality Assurance | This document has been electronica
  first relevant chunk at rank 1 of 12
  ANSWER: The following quality control test methods and criteria were used:

* **Flow Rate Test:** Performed according to SOP-QC-042.
* **Pressure Integrity:** Tested at a maximum of 0.5 bar with a 5-minute hold.
* **Visual Inspection:** Examined for visible defects.
* **Package Integrity:** Evaluated to ensure packaging is seal


--- Q3: Which bio-safety and regulatory standards do the materials comply with?
  [1] score=0.5131 page=2 (RELEVANT)
      Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulatory Conformance  | Bio-safety | All polymeric materials in contact with the process fluid comply with United State
  [2] score=0.4168 page=3 (RELEVANT)
      0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visual Inspection | No defects visible | Pass | Package Integrity | Sealed, no damage | Pass | Regulatory Conformance | Bio-safet
  first relevant chunk at rank 1 of 12
  ANSWER: All polymeric materials in contact with the process fluid comply with the following bio-safety and regulatory standards:

* **United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI**
* **EMA/410/01** (or are free from animal-derived ingredients)

E5: intfloat/e5-small-v2


...

query prefix: 'Represent this sentence for searching relevant passages: ' | passage prefix: None



--- Q1: What are the storage conditions for this product?
  [1] score=0.6613 page=1 (RELEVANT)
      Cytiva | 100 Results Way | Marlborough, MA 01752 | United States | Page 1 / 1 | cytiva.com | 3 June, 2022 | Re: ÄKTATM ready Flow Kit Storage Conditions | To Whom It May Concern, | The recommended storage temperature for standard ÄKTA rea
  [2] score=0.6125 page=1 (not relevant)
      Extended storage below the recommended +5 | could lead to  | brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2 C to  | +40 C. If the kits are allowed to acclima
  first relevant chunk at rank 1 of 12
  ANSWER: The recommended storage temperature for standard and modified ÄKTA ready flow kits is greater than +5 °C (> +5 °C). Extended storage below +5 °C should be avoided, as it could lead to brittleness or cracking of the plastic connectors.


  Gemini call failed (ClientError); retrying in 10s



--- Q2: What test methods were used for quality control?
  [1] score=0.5804 page=3 (RELEVANT)
      0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visual Inspection | No defects visible | Pass | Package Integrity | Sealed, no damage | Pass | Regulatory Conformance | Bio-safet
  [2] score=0.5336 page=3 (not relevant)
      Cytiva | cytiva.com | Certificate of Quality | This product is manufactured in compliance with our ISO 9001 certified quality management system. | Issued by Cytiva Westborough Quality Assurance | This document has been electronica
  first relevant chunk at rank 1 of 12
  ANSWER: The test methods used for quality control include:

* **Flow Rate Test:** Performed in accordance with SOP-QC-042.
* **Pressure Integrity Test:** Evaluated with a 5-minute hold at a maximum of 0.5 bar.
* **Visual Inspection:** Conducted to ensure no visible defects are present.
* **Package Integrity Test:** Performed to


--- Q3: Which bio-safety and regulatory standards do the materials comply with?
  [1] score=0.7333 page=2 (RELEVANT)
      Description  | Limits  | Autoclave | Pump tubing | 121°C > 15 min  | Gamma Irradiation  | Inlets  | 25.0 – 40.0 kGy  | Regulatory Conformance  | Bio-safety | All polymeric materials in contact with the process fluid comply with United State
  [2] score=0.7181 page=3 (RELEVANT)
      0 - 40.0 kGy | Conforms | Flow Rate Test | Per SOP-QC-042 | Pass | Pressure Integrity | Max 0.5 bar, 5 min hold | Pass | Visual Inspection | No defects visible | Pass | Package Integrity | Sealed, no damage | Pass | Regulatory Conformance | Bio-safet
  first relevant chunk at rank 1 of 12
  ANSWER: All polymeric materials in contact with the process fluid comply with the following bio-safety and regulatory standards:

* **United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI**
* **EMA/410/01** (or are free from animal-derived ingredients)

Done.


## Step 6: Scorecard

- **Hit@k**: number of queries where the answer chunk was inside the top-k handed to the LLM
- **MRR**: mean reciprocal rank of the first relevant chunk (1.0 = always ranked first)
- **Score margin**: gap between the similarity of the top chunk and the second chunk, averaged over queries. A larger margin means the model separates the best chunk more decisively. Raw similarity scores are *not* comparable across models (each has its own score distribution), which is why the table reports ranks and margins rather than raw scores.

In [7]:
import pandas as pd

rows = []
for label, r in results.items():
    qs = r["queries"]
    rows.append({
        "Model": label,
        f"Hit@{TOP_K}": f"{sum(q['hit'] for q in qs)}/{len(qs)}",
        "MRR": round(sum(1 / q["first_relevant_rank"] for q in qs) / len(qs), 3),
        **{f"{q['id']} rank": q["first_relevant_rank"] for q in qs},
        "Top-1 relevant": f"{sum(q['chunks'][0]['relevant'] for q in qs)}/{len(qs)}",
        "Score margin": round(sum(q["chunks"][0]["score"] - q["chunks"][1]["score"] for q in qs) / len(qs), 4),
        "Index (s)": round(r["index_s"], 2),
        "Query (ms)": round(sum(q["query_ms"] for q in qs) / len(qs), 1),
        "Load (s)": round(r["load_s"], 1),
    })

scorecard = pd.DataFrame(rows).set_index("Model")
scorecard

,Hit@2,MRR,Q1 rank,Q2 rank,Q3 rank,Top-1 relevant,Score margin,Index (s),Query (ms),Load (s)
Model,,,,,,,,,,
MiniLM,3/3,1.0,1,1,1,3/3,0.0738,0.17,14.1,1.3
E5,3/3,1.0,1,1,1,3/3,0.0022,0.23,18.9,1.6
BGE,3/3,1.0,1,1,1,3/3,0.0369,0.24,22.3,1.4


## Step 7: Generated answers side by side

In [8]:
if USE_LLM:
    for i, q in enumerate(QUERIES):
        print(f"\n{'=' * 90}\n{q['id']}: {q['query']}\n{'=' * 90}")
        for label, r in results.items():
            print(f"\n[{label}]\n{r['queries'][i]['answer']}")
else:
    print("Retrieval-only run (no Gemini key). Add a key in Step 1 and re-run to generate answers.")


Q1: What are the storage conditions for this product?

[MiniLM]
The recommended storage temperature is greater than +5°C (> +5°C).

[E5]
The recommended storage temperature for ÄKTA ready flow kits is greater than +5 °C (> +5 °C). Extended storage below +5 °C should be avoided, as it could lead to brittleness or cracking of the plastic connectors.

[BGE]
The recommended storage temperature for standard and modified ÄKTA ready flow kits is greater than +5 °C (> +5 °C). Extended storage below +5 °C should be avoided, as it could lead to brittleness or cracking of the plastic connectors.

Q2: What test methods were used for quality control?

[MiniLM]
The following quality control test methods and criteria were used:

* **Flow Rate Test:** Performed according to SOP-QC-042.
* **Pressure Integrity:** Tested at a maximum of 0.5 bar with a 5-minute hold.
* **Visual Inspection:** Examined for visible defects.
* **Package Integrity:** Evaluated to ensure packaging is sealed with no damage.
* *

## Step 8: Paste-ready text for the comparison document

Prints the query, generated answer and retrieved chunks for each model in the layout of the "Embedding Model Comparison Results" template.

In [9]:
REPORT_QUERY = 0  # which query to show in the per-model sections (0 = Q1)

for label, r in results.items():
    q = r["queries"][REPORT_QUERY]
    print(f"\n{'#' * 90}\nModel: {label}\nModel Name: {r['model_name']}\n")
    print(f"Query Tested:\n{q['query']}\n")
    print(f"Generated Answer:\n{q['answer'] or '(retrieval-only run)'}\n")
    print("Retrieved Chunks:")
    for c in q["chunks"]:
        print(f"  Chunk {c['rank']} (page {c['page']}, similarity {c['score']:.4f}):")
        print("  " + c["text"].replace("\n", "\n  "))
        print()


##########################################################################################
Model: MiniLM
Model Name: sentence-transformers/all-MiniLM-L6-v2

Query Tested:
What are the storage conditions for this product?

Generated Answer:
The recommended storage temperature is greater than +5°C (> +5°C).

Retrieved Chunks:
  Chunk 1 (page 1, similarity 0.2776):
  Cytiva
  100 Results Way
  Marlborough, MA 01752
  United States
  Page 1 / 1
  cytiva.com
  3 June, 2022
  Re: ÄKTATM ready Flow Kit Storage Conditions
  To Whom It May Concern,
  The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
  Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
  as well, including the two listed in the below table.

  Chunk 2 (page 2, similarity 0.1866):
  Description 
  Limits 
  Autoclave
  Pump tubing
  121°C > 15 min 
  Gamma Irradiation 
  Inlets 
  25.0 – 40.0 kGy 
 

## Findings

Results below are from a CPU run of this notebook (12 chunks, `top_k=2`). Timings will differ between machines; rankings will not.

| Model | Right chunk ranked #1 | Hit@2 | Avg. gap between #1 and #2 score | Index time | Query time |
|---|---|---|---|---|---|
| MiniLM | 3/3 | 3/3 | 0.074 | 0.18 s | ~12 ms |
| E5 | 3/3 | 3/3 | 0.002 | 0.24 s | ~16 ms |
| BGE | 3/3 | 3/3 | 0.037 | 0.24 s | ~19 ms |

**1. All three models find the right chunk.** On a 3-page document with clearly worded queries, each model ranked the answer chunk first for all three queries. The headline metric does not separate them, so the differences are in the details below.

**2. The second chunk is where they differ.** For Q1 (storage conditions), E5 and BGE returned the two halves of the storage letter: the "> +5 C" recommendation, plus the follow-on chunk explaining that storage below +5 risks brittle connectors and that the operating range is +2 C to +40 C. MiniLM returned the right first chunk but its second was the page-2 certificate's autoclave/irradiation limits, which has nothing to do with storage. That showed up in the generated answers: from MiniLM's context Gemini answered only "greater than +5 C", while from E5's and BGE's context it also warned that extended storage below +5 C can make the plastic connectors brittle or crack. All three answers were correct; E5's and BGE's were more complete. For Q2 and Q3 the three models produced effectively the same answer.

**3. E5 only works properly with its prefixes.** With a plain `model_name` swap (no `query: ` / `passage: ` prefixes), E5 ranked the storage answer chunk *second* for Q1, behind the follow-on chunk. Adding the prefixes moved it to first. This is the main practical lesson: an embedding model can look worse than it is if it is not called the way it was trained.

**4. E5's scores are tightly packed.** E5's similarities all sit around 0.78 to 0.84, and the gap between its first and second chunk averages 0.002. It ranks correctly, but with almost no margin, so a fixed similarity cutoff would be hard to set for it. MiniLM (0.19 to 0.51) and BGE (0.53 to 0.73) spread their scores much more. This matters for the next step, where similarity thresholds are tuned.

**5. Speed is not a deciding factor here.** MiniLM is the fastest (it is the smallest model), but on 12 chunks every model indexes in about a quarter of a second and retrieves in under 20 ms. The difference would only matter on a much larger corpus.

**6. A chunking limit that no embedding model fixes.** For Q2 (test methods), the test table on page 3 was split across two chunks: the Autoclave and Gamma Irradiation rows landed in one chunk, and Flow Rate, Pressure Integrity, Visual Inspection and Package Integrity in the next. Every model retrieved the second of these and none retrieved the first in its top 2, and the generated answers from all three models list those four tests but leave out Autoclave and Gamma Irradiation. That is a chunk-size / `top_k` problem, not an embedding problem, and is the first thing to address in the retriever-tuning task.

**Choice: BGE (`BAAI/bge-small-en-v1.5`).** It ranked the right chunk first on every query, pulled the most useful supporting chunk for the storage question, and keeps a clear score margin between good and weaker chunks, at a speed cost that is negligible at this scale. MiniLM is a reasonable choice if speed is the priority; E5 is accurate once prefixed but its compressed scores make it the hardest to tune.

**Limits of this test.** Three queries on one 3-page document is a small sample, and all three models scored perfectly on ranking, so the choice rests on secondary evidence (second-chunk quality and score separation). A larger document set with harder queries would be needed to claim one model is better in general.